# Clean ECG signal images on EC2

This standalone notebook reads the patient-disjoint `ECG_500Hz_patient_splits.tar.gz` archive and renders **one clean image per ECG record** with the real ECG-image-kit (vendored `ecg_image_v1_code.tar.gz`, dataset_generation_code PLAN §6). It does not filter, resample, normalize, synthesize, or augment waveform samples. Each record is staged to a canonical scratch WFDB `.dat` (fmt 16, gain 1000, canonical lead order, int16 round-trip verified — this dodges the kit's `.mat` gain bug, PLAN §0.5) and rendered by the kit's `gen_ecg_images_from_data_batch.py` with the reference flags (`--num_columns 4 --full_mode II -r 200 --pad_inches 0 --calibration_pulse 1 --random_bw 1`, `ECG_ROW_GAP=4.4 ECG_RHYTHM_GAP=5.5 ECG_BOTTOM_RESERVE=1.9`, fixed `Arial.ttf`): standard clinical 3x4 layout (2.5 s per lead panel) plus a continuous full 10 s Lead II rhythm strip at 25 mm/s, 10 mm/mV, with calibration pulses and lead names and no printed ID header. The grey 2200x1700 master is recoloured to the canonical clean red grid (fixed LUT, PLAN §10) and resized with LANCZOS to 994x768 plus 15 px paper-colour pads on each side, giving a final 1024x768 PNG. Pixel rasterization plus the fixed recolour/resize is the only lossy step.

Run cells in order on Linux. Copy **this notebook, `ecg_image_v1_code.tar.gz`, and `ecg_image_v1_code.tar.gz.sha256`** to the same folder on EC2, then launch Jupyter from that folder. The notebook verifies the local kit bundle against its SHA-256 sidecar; if no local copy is found, it tries `S3_KIT_SOURCE` (§1) when those two kit objects have been uploaded there. It downloads the source archive and its `.sha256` sidecar, verifies the complete archive before extraction, verifies the kit bundle against its sidecar, refuses to render unless every `patch_kit` marker is present, and writes its own worker module into the configured working directory. The output mirrors `train/`, `validation/`, and `test/` under `images/`. The package includes images, the source patient and record manifests, a rendering manifest, a failure log, a configuration snapshot, and a summary. Reruns reuse valid existing images when rendering settings match.

**Expected source layout:** `ECG_500Hz_patient_splits/{train,validation,test}/I0001/WFDB/.../*.hea` and corresponding `I0006/WFDB/` records. Signal companions are `.mat` or `.dat`. Every record has 12 leads, 5,000 samples at 500 Hz; each record is validated independently before staging.


In [ ]:
# 1. Configuration: edit before running any later cell.
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

from pathlib import Path

S3_SOURCE = "s3://ecg-thesis-share/ECG_500Hz_patient_splits.tar.gz"
S3_SHA256_SOURCE = "s3://ecg-thesis-share/ECG_500Hz_patient_splits.tar.gz.sha256"
S3_KIT_SOURCE = "s3://ecg-thesis-share/ecg_image_v1_code.tar.gz"
S3_KIT_SHA256_SOURCE = "s3://ecg-thesis-share/ecg_image_v1_code.tar.gz.sha256"
WORK_DIR = Path.home() / "ecg_clean_render"
LOCAL_ARCHIVE = WORK_DIR / "input" / "ECG_500Hz_patient_splits.tar.gz"
INPUT_DIR = WORK_DIR / "input" / "ECG_500Hz_patient_splits"
OUTPUT_DIR = WORK_DIR / "clean_ecg_signal_images_kit_v1"
S3_UPLOAD_PREFIX = "s3://ecg-thesis-share/generated_images/"
ARCHIVE_FORMAT = "tar"            # "tar" (fast; PNG is compressed) or "tar.gz"
ARCHIVE_BASENAME = "clean_ecg_signal_images_kit_v1"

WORKERS = min(14, max(1, (os.cpu_count() or 2) - 2))  # 14 on c8a.4xlarge
BATCH_SIZE = 24                    # records per kit invocation; ~9 s/record
MAX_PENDING_BATCHES = 2 * WORKERS  # bounds task/result memory
PROGRESS_EVERY = 1000

# Rendering uses the real ECG-image-kit (PLAN §6): 200-DPI grey masters
# (2200x1700 px), 3x4 (2.5 s/lead) + full 10 s Lead II rhythm strip,
# 25 mm/s, 10 mm/mV, calibration pulses, fixed Arial; recoloured to the
# canonical clean red grid and resized to a final 1024x768 PNG (PLAN §7.3).
IMAGE_FORMAT = "PNG"
FINAL_IMAGE_WIDTH = 1024
FINAL_IMAGE_HEIGHT = 768
PNG_COMPRESS_LEVEL = 3             # fast lossless PNG
MAX_DURATION_S = 10.0
KIT_SEED = 20260918                # kit -se (dead code in the kit; passed anyway)
KIT_FIXED_FONT = "Arial.ttf"       # clean images, every split (PLAN §6.7)
KIT_RENDER_FLAGS = ["--num_columns", "4", "--full_mode", "II", "-r", "200",
                    "--pad_inches", "0", "--calibration_pulse", "1",
                    "--random_bw", "1"]  # grey master; recoloured clean-red after
KIT_ENV = {"ECG_ROW_GAP": "4.4", "ECG_RHYTHM_GAP": "5.5",
           "ECG_BOTTOM_RESERVE": "1.9"}
RENDER_VERSION = 4  # 4 = real ECG-image-kit + clean-red recolour + 1024x768

assert IMAGE_FORMAT == "PNG", "This implementation produces lossless PNG files"
assert ARCHIVE_FORMAT in {"tar", "tar.gz"}
assert WORKERS >= 1 and BATCH_SIZE >= 1 and MAX_PENDING_BATCHES >= WORKERS
assert (FINAL_IMAGE_WIDTH, FINAL_IMAGE_HEIGHT) == (1024, 768)
assert 1 <= PNG_COMPRESS_LEVEL <= 9 and MAX_DURATION_S > 0
print("Working directory:", WORK_DIR)
print("Workers:", WORKERS, "| archive:", ARCHIVE_BASENAME + "." + ARCHIVE_FORMAT)
print("Kit flags:", " ".join(KIT_RENDER_FLAGS), "| font:", KIT_FIXED_FONT, "| env:", KIT_ENV)


## 2. Install and verify all runtime dependencies

The third-party packages this notebook and its generated worker need are `boto3`, `numpy`, `scipy`, and `Pillow` (worker staging + recolour + resize), plus `matplotlib`, `pandas`, `qrcode`, `pyyaml`, and `wfdb` (the kit's batch-driver imports — `wfdb>=4.3` is REQUIRED: 4.1.x dies at import on pandas 3). The next cell installs every required package up front with `%pip` (unconditional — a fast no-op when already satisfied, so a half-present environment can never slip through and fail later at an `import`), upgrades a stale `wfdb` before it is ever imported, and then verifies each import with an automatic repair install for any straggler.

Before downloading, attach an **EC2 IAM role (instance profile)** to this instance. The role needs `s3:GetObject` on the dataset archive and its `.sha256` sidecar. To upload and verify the output, it also needs `s3:PutObject` and `s3:GetObject` on `ecg-thesis-share/generated_images/*`; `s3:AbortMultipartUpload` and `s3:ListMultipartUploadParts` support cleanup and retries for the large archive. If you get `NoCredentialsError`, use EC2 console → Instances → select this instance → Actions → Security → Modify IAM role, attach the role, then rerun the S3 cell. KMS-encrypted objects additionally require the applicable KMS permissions. The bucket must be reachable from the instance's region/network.


In [ ]:
# Install ALL runtime packages into this kernel's environment BEFORE any
# third-party import. Unconditional (a fast no-op when already satisfied) so a
# half-present environment can never slip through a presence check and fail
# later at `import scipy` with ModuleNotFoundError.
%pip install --quiet boto3 numpy scipy Pillow matplotlib pandas qrcode pyyaml wfdb
import importlib as _package_importlib
import subprocess as _package_subprocess
import sys as _kernel_sys

# boto3 = S3 I/O; numpy/scipy/Pillow = worker staging + recolour + resize;
# matplotlib/pandas/qrcode/pyyaml/wfdb = the kit batch-driver imports.
_module_to_spec = {"boto3": "boto3", "numpy": "numpy", "scipy": "scipy",
                   "PIL": "Pillow", "matplotlib": "matplotlib",
                   "pandas": "pandas", "qrcode": "qrcode",
                   "yaml": "pyyaml", "wfdb": "wfdb>=4.3"}

# wfdb>=4.3 is REQUIRED (4.1.x dies at import on pandas 3): upgrade a stale
# copy BEFORE it is ever imported in this process (upgrading after a failed
# import mixes versions and needs a kernel restart instead).
try:
    from importlib.metadata import version as _dist_version
    _wfdb_have = _dist_version("wfdb")
    _wfdb_old = tuple(int(p) for p in _wfdb_have.split(".")[:2]) < (4, 3)
except Exception:
    _wfdb_have, _wfdb_old = "?", False
if _wfdb_old:
    if "wfdb" in _kernel_sys.modules:
        raise RuntimeError(
            f"wfdb {_wfdb_have} is already imported and too old (need >=4.3): "
            "restart the kernel and re-run from §1.")
    print(f"Upgrading stale wfdb {_wfdb_have} to wfdb>=4.3 ...", flush=True)
    _package_subprocess.check_call(
        [_kernel_sys.executable, "-m", "pip", "install", "--quiet",
         "--retries", "5", "--timeout", "300", "wfdb>=4.3"])
    _package_importlib.invalidate_caches()

# Verify every import; an unimportable module gets one automatic repair
# install (covers detection gaps and broken copies) instead of failing later.
for _mod, _spec in _module_to_spec.items():
    try:
        _package_importlib.import_module(_mod)
    except Exception as exc:
        print(f"{_mod} not importable ({type(exc).__name__}); "
              f"repair-installing {_spec} ...", flush=True)
        _package_subprocess.check_call(
            [_kernel_sys.executable, "-m", "pip", "install", "--quiet",
             "--retries", "5", "--timeout", "300", _spec])
        _package_importlib.invalidate_caches()
        _package_importlib.import_module(_mod)
print("Runtime packages verified:",
      ", ".join(sorted(_module_to_spec)), flush=True)

import csv
import hashlib
import importlib
import json
import math
import multiprocessing as mp
import shutil
import sys
import tarfile
import time
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
from datetime import datetime, timezone
from urllib.parse import urlparse

import boto3
import numpy as np
import scipy
from scipy.io import loadmat as _checked_loadmat
from PIL import Image
from boto3.s3.transfer import TransferConfig

print("Python", sys.version.split()[0], "| boto3", boto3.__version__,
      "| NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| Pillow", Image.__version__)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Download, verify, and extract

The dataset archive and its SHA-256 checksum are read from S3. The downloaded archive is hashed in full before extraction. A matching local archive can be reused. Extraction rejects paths outside the destination and all links and special files. A completion marker allows a clean restart. Keep enough EBS space for the downloaded archive and extracted tree at the same time.


In [ ]:
def split_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc or ":" in parsed.netloc or not parsed.path.strip("/"):
        raise ValueError(f"Expected s3://bucket/key, got {uri!r}")
    return parsed.netloc, parsed.path.lstrip("/")

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            block = stream.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

from botocore.exceptions import NoCredentialsError, PartialCredentialsError

aws_session = boto3.Session()
try:
    aws_credentials = aws_session.get_credentials()
except (NoCredentialsError, PartialCredentialsError) as exc:
    aws_credentials = None
if aws_credentials is None:
    raise RuntimeError(
        "No AWS credentials are available to this Jupyter kernel. Attach an EC2 IAM "
        "role to this instance: EC2 console > Instances > select instance > Actions > "
        "Security > Modify IAM role. Grant read access to the dataset archive and "
        "checksum, plus write/read access under generated_images/. Then rerun this cell."
    )
print("AWS credential source:", aws_credentials.method)
s3 = aws_session.client("s3")
src_bucket, src_key = split_s3_uri(S3_SOURCE)
sha_bucket, sha_key = split_s3_uri(S3_SHA256_SOURCE)
try:
    checksum_object = s3.get_object(Bucket=sha_bucket, Key=sha_key)
except (NoCredentialsError, PartialCredentialsError) as exc:
    raise RuntimeError(
        "AWS credentials could not be used. Check the EC2 instance IAM role and "
        "instance metadata access, then rerun this cell."
    ) from exc
if int(checksum_object["ContentLength"]) > 1024:
    raise ValueError("S3 SHA-256 sidecar is unexpectedly large")
checksum_fields = checksum_object["Body"].read().decode("utf-8").strip().split()
if len(checksum_fields) != 2 or checksum_fields[1].lstrip("*") != LOCAL_ARCHIVE.name:
    raise ValueError("SHA-256 sidecar must name the dataset archive")
expected_sha256 = checksum_fields[0].lower()
if len(expected_sha256) != 64 or any(char not in "0123456789abcdef" for char in expected_sha256):
    raise ValueError("SHA-256 sidecar does not contain a valid digest")

src_info = s3.head_object(Bucket=src_bucket, Key=src_key)
source_bytes = int(src_info["ContentLength"])
source_identity = {"uri": S3_SOURCE, "sha256_uri": S3_SHA256_SOURCE,
                   "sha256": expected_sha256, "bytes": source_bytes,
                   "etag": src_info.get("ETag", "")}
source_sidecar = LOCAL_ARCHIVE.with_name(LOCAL_ARCHIVE.name + ".source.json")
archive_matches = (LOCAL_ARCHIVE.is_file() and LOCAL_ARCHIVE.stat().st_size == source_bytes
                   and file_sha256(LOCAL_ARCHIVE) == expected_sha256)
if archive_matches:
    print("Reusing SHA-256 verified source archive:", LOCAL_ARCHIVE)
else:
    partial = LOCAL_ARCHIVE.with_name(LOCAL_ARCHIVE.name + ".part")
    partial.unlink(missing_ok=True)
    print(f"Downloading {S3_SOURCE} ({source_bytes / 2**30:.2f} GiB) ...")
    s3.download_file(src_bucket, src_key, str(partial),
                     Config=TransferConfig(max_concurrency=8, use_threads=True))
    if partial.stat().st_size != source_bytes:
        raise IOError("Downloaded byte count differs from S3 ContentLength")
    if file_sha256(partial) != expected_sha256:
        partial.unlink(missing_ok=True)
        raise IOError("Downloaded archive SHA-256 does not match S3 sidecar")
    partial.replace(LOCAL_ARCHIVE)
    print("Downloaded and SHA-256 verified:", LOCAL_ARCHIVE)
source_sidecar.write_text(json.dumps(source_identity, indent=2) + "\n")

extraction_root = INPUT_DIR.parent.resolve()
marker = extraction_root / ".ecg_patient_splits_extraction_complete.json"
marker_value = {"source_uri": S3_SOURCE, "archive_sha256": expected_sha256,
                "archive_bytes": source_bytes,
                "archive_mtime": LOCAL_ARCHIVE.stat().st_mtime_ns}
already_extracted = False
if marker.is_file():
    try:
        already_extracted = json.loads(marker.read_text()) == marker_value and INPUT_DIR.is_dir()
    except (OSError, ValueError):
        pass
if already_extracted:
    print("Reusing extracted patient-split ECG tree:", INPUT_DIR)
else:
    marker.unlink(missing_ok=True)
    print("Extracting to", extraction_root, "...")
    member_count = 0
    from pathlib import PurePosixPath
    with tarfile.open(LOCAL_ARCHIVE, "r:gz") as archive:
        for member in archive:
            rel = PurePosixPath(member.name)
            if rel.is_absolute() or ".." in rel.parts or not rel.parts or rel.parts[0] != INPUT_DIR.name:
                raise ValueError(f"Unsafe/unexpected archive path: {member.name!r}")
            dest = extraction_root.joinpath(*rel.parts)
            if not dest.resolve(strict=False).is_relative_to(extraction_root):
                raise ValueError(f"Archive path resolves outside destination: {member.name!r}")
            if member.isdir():
                dest.mkdir(parents=True, exist_ok=True)
            elif member.isfile():
                if dest.is_symlink():
                    raise ValueError(f"Refusing to overwrite symlink: {dest}")
                dest.parent.mkdir(parents=True, exist_ok=True)
                stream = archive.extractfile(member)
                if stream is None:
                    raise IOError(f"Cannot read archive member {member.name}")
                with stream, dest.open("wb") as out:
                    shutil.copyfileobj(stream, out, length=1024 * 1024)
                if dest.stat().st_size != member.size:
                    raise IOError(f"Incomplete extracted member {member.name}")
                member_count += 1
            else:
                raise ValueError(f"Links/special files are disallowed: {member.name!r}")
    if not INPUT_DIR.is_dir():
        raise FileNotFoundError(f"Archive did not create {INPUT_DIR}")
    marker.write_text(json.dumps(marker_value, indent=2) + "\n")
    print(f"Extracted {member_count:,} files")


## 4. Kit bundle: verify, extract, patch-gate

The `ecg_image_v1_code.tar.gz` bundle (plus its `.sha256` sidecar) is hashed in full before extraction — the same sidecar-pinned discipline as the source archive. Rendering refuses to start unless every `patch_kit` marker is present and the fixed `Arial.ttf` ships in `Fonts/`. A completion marker allows a clean restart. Candidates, in order: the Jupyter working directory (launch Jupyter from the folder holding this notebook and the two kit files), `WORK_DIR/bundle/`, then an available `dataset_generation_code/` copy.
If no local copy exists, the bundle is pulled from S3 (`S3_KIT_SOURCE`, §1) when you have uploaded both kit objects there.


In [ ]:
KIT_BUNDLE = None
_kit_search = [Path.cwd() / "ecg_image_v1_code.tar.gz",
               WORK_DIR / "bundle" / "ecg_image_v1_code.tar.gz",
               WORK_DIR / "ecg_image_v1_code.tar.gz"]
for _ancestor in [Path.cwd(), *list(Path.cwd().parents)[:4]]:
    _kit_search.append(_ancestor / "dataset_generation_code" / "ecg_image_v1_code.tar.gz")
for _cand in _kit_search:
    if _cand.is_file():
        KIT_BUNDLE = _cand
        break
if KIT_BUNDLE is None:
    # No local copy — pull the kit bundle from S3 with the same sidecar-pinned
    # discipline as the source archive, then fall through to the checks below.
    try:
        print("No local kit bundle; downloading from", S3_KIT_SOURCE, "...", flush=True)
        _kit_dest = WORK_DIR / "bundle" / "ecg_image_v1_code.tar.gz"
        _kit_dest.parent.mkdir(parents=True, exist_ok=True)
        _kb, _kk = split_s3_uri(S3_KIT_SOURCE)
        _sb, _sk = split_s3_uri(S3_KIT_SHA256_SOURCE)
        _sobj = s3.get_object(Bucket=_sb, Key=_sk)
        if int(_sobj["ContentLength"]) > 1024:
            raise ValueError("Kit SHA-256 sidecar is unexpectedly large")
        _raw = _sobj["Body"].read()
        (_kit_dest.parent / (_kit_dest.name + ".sha256")).write_bytes(_raw)
        _fields = _raw.decode("utf-8").strip().split()
        if len(_fields) != 2 or _fields[1].lstrip("*") != _kit_dest.name:
            raise ValueError("Kit SHA-256 sidecar must name the kit bundle")
        _want = _fields[0].lower()
        if len(_want) != 64 or any(c not in "0123456789abcdef" for c in _want):
            raise ValueError("Kit SHA-256 sidecar does not contain a valid digest")
        _kinfo = s3.head_object(Bucket=_kb, Key=_kk)
        _kbytes = int(_kinfo["ContentLength"])
        if not (_kit_dest.is_file() and _kit_dest.stat().st_size == _kbytes
                and file_sha256(_kit_dest) == _want):
            _kpart = _kit_dest.with_name(_kit_dest.name + ".part")
            _kpart.unlink(missing_ok=True)
            print(f"Downloading {S3_KIT_SOURCE} ({_kbytes / 2**20:.1f} MiB) ...", flush=True)
            s3.download_file(_kb, _kk, str(_kpart),
                             Config=TransferConfig(max_concurrency=8, use_threads=True))
            if _kpart.stat().st_size != _kbytes:
                raise IOError("Downloaded kit byte count differs from S3 ContentLength")
            if file_sha256(_kpart) != _want:
                _kpart.unlink(missing_ok=True)
                raise IOError("Downloaded kit SHA-256 does not match S3 sidecar")
            _kpart.replace(_kit_dest)
            print("Downloaded and SHA-256 verified:", _kit_dest, flush=True)
        else:
            print("Reusing SHA-256 verified kit bundle:", _kit_dest, flush=True)
        KIT_BUNDLE = _kit_dest
    except Exception as exc:
        raise FileNotFoundError(
            "ecg_image_v1_code.tar.gz not found locally and the S3 fallback failed "
            f"({type(exc).__name__}: {exc}). Searched: " +
            ", ".join(str(p) for p in _kit_search) +
            f"; S3: {S3_KIT_SOURCE}") from exc
_kit_sidecar = Path(str(KIT_BUNDLE) + ".sha256")
if not _kit_sidecar.is_file():
    raise FileNotFoundError(f"Kit SHA-256 sidecar missing: {_kit_sidecar}")
_kit_want = _kit_sidecar.read_text().split()[0].lower()
if len(_kit_want) != 64 or any(c not in "0123456789abcdef" for c in _kit_want):
    raise ValueError("Kit SHA-256 sidecar does not contain a valid digest")
KIT_BUNDLE_SHA256 = file_sha256(KIT_BUNDLE)
if KIT_BUNDLE_SHA256 != _kit_want:
    raise IOError("Kit bundle SHA-256 does not match its sidecar — re-transfer the file")
print("Kit bundle verified:", KIT_BUNDLE)

KIT_ROOT = WORK_DIR / "kit"
_kit_marker = KIT_ROOT / ".kit_extraction_complete.json"
_kit_marker_value = {"bundle_sha256": KIT_BUNDLE_SHA256,
                     "bundle_bytes": KIT_BUNDLE.stat().st_size}
_kit_ready = False
if _kit_marker.is_file():
    try:
        _kit_ready = json.loads(_kit_marker.read_text()) == _kit_marker_value and KIT_ROOT.is_dir()
    except (OSError, ValueError):
        pass
if _kit_ready:
    print("Reusing extracted kit:", KIT_ROOT)
else:
    _kit_marker.unlink(missing_ok=True)
    print("Extracting kit to", KIT_ROOT, "...")
    shutil.rmtree(KIT_ROOT, ignore_errors=True)
    with tarfile.open(KIT_BUNDLE, "r:gz") as _archive:
        try:
            _archive.extractall(KIT_ROOT, filter="data")
        except TypeError:  # Python without PEP 706 filters; bundle is SHA-256 verified
            _archive.extractall(KIT_ROOT)
    _kit_marker.write_text(json.dumps(_kit_marker_value, indent=2) + "\n")
    print("Kit extracted")

KIT_DIR = str(KIT_ROOT / "ecg_image_v1_code" / "ecg-image-generator")
_KIT_SCRIPT = Path(KIT_DIR) / "gen_ecg_images_from_data_batch.py"
if not _KIT_SCRIPT.is_file():
    raise FileNotFoundError(f"Kit batch driver missing: {_KIT_SCRIPT}")
if not (Path(KIT_DIR) / "Fonts" / KIT_FIXED_FONT).is_file():
    raise FileNotFoundError(f"Fixed font {KIT_FIXED_FONT} missing from kit Fonts/")

# Patch gate (PLAN §6): refuse to render from unpatched kit code.
_kit_need = {
    "ecg_plot.py": ["PATCHED by image_pipeline/patch_kit.py",
                    "PATCHED-LEADNAME", "PATCHED-RHYTHMNAME",
                    "PATCHED-INKWIDTH", "PATCHED-LAYOUT", "PATCHED-PAPER"],
    "extract_leads.py": ["PATCHED-COLUMNS"],
    "gen_ecg_images_from_data_batch.py": ["PATCHED-NUMPY2"],
    "gen_ecg_image_from_data.py": ["PATCHED-LAZYDEPS", "PATCHED-FIXEDFONT"],
    "helper_functions.py": ["PATCHED-UNUSED-IMGAUG"],
}
_kit_missing = []
for _name, _markers in _kit_need.items():
    _text = (Path(KIT_DIR) / _name).read_text()
    for _mk in _markers:
        if _mk not in _text:
            _kit_missing.append(f"{_name}:{_mk}")
if _kit_missing:
    raise RuntimeError(f"Kit at {KIT_DIR} is not patched "
                       f"({len(_kit_missing)} markers missing, e.g. {_kit_missing[:3]}); refusing to render")
print(f"Kit patch gate OK ({KIT_DIR})")


## 5. Discover and validate split records

The archive's patient and record manifests are checked against the extracted headers before rendering. Input paths retain their `train/`, `validation/`, or `test/` prefix. Output PNG paths mirror those prefixes under `images/`, and the patient mapping files are copied into the image package for later audits or patient-level re-splitting.


In [ ]:
headers = sorted(INPUT_DIR.rglob("*.hea"))
if not headers:
    raise FileNotFoundError(f"No WFDB .hea files found under {INPUT_DIR}")
records = [p.relative_to(INPUT_DIR).as_posix() for p in headers]
if len(set(records)) != len(records):
    raise RuntimeError("Duplicate relative WFDB header paths")

splits = {"train", "validation", "test"}
patient_map = {}
with (INPUT_DIR / "patient_assignments.csv").open(newline="", encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        patient = row["patient_key"]
        if patient in patient_map or row["split"] not in splits:
            raise ValueError(f"Duplicate patient or unknown split: {patient}")
        patient_map[patient] = row["split"]

manifest_paths = set()
split_counts = {split: 0 for split in splits}
with (INPUT_DIR / "record_manifest.csv").open(newline="", encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        split = row["split"]
        rel = row["split_header_path"]
        if split not in splits or rel != f"{split}/{row['record_path']}":
            raise ValueError(f"Invalid split path in record manifest: {rel}")
        if patient_map.get(row["patient_key"]) != split:
            raise ValueError(f"Patient split conflict in record manifest: {row['patient_key']}")
        if rel in manifest_paths:
            raise ValueError(f"Duplicate record in manifest: {rel}")
        manifest_paths.add(rel)
        split_counts[split] += 1
if manifest_paths != set(records):
    raise ValueError("Extracted WFDB headers do not match the patient-split record manifest")
split_audit = json.loads((INPUT_DIR / "split_audit.json").read_text())
if (split_audit.get("status") != "validated"
        or split_audit.get("record_counts") != split_counts
        or split_audit.get("patients_total") != len(patient_map)):
    raise ValueError("Split audit does not match extracted records")
print(f"Discovered {len(records):,} WFDB headers from {len(patient_map):,} patients")
print("Split records:", split_counts)

for name in ("record_manifest.csv", "patient_assignments.csv", "split_audit.json", "README.md"):
    shutil.copy2(INPUT_DIR / name, OUTPUT_DIR / name)

render_config = {
    "render_version": RENDER_VERSION,
    "renderer": "ecg-image-kit",
    "source_uri": S3_SOURCE,
    "source_sha256": expected_sha256,
    "source_etag": source_identity["etag"],
    "source_bytes": source_bytes,
    "split_record_counts": split_counts,
    "kit_bundle_sha256": KIT_BUNDLE_SHA256,
    "kit_seed": KIT_SEED,
    "kit_fixed_font": KIT_FIXED_FONT,
    "kit_render_flags": KIT_RENDER_FLAGS,
    "kit_env": KIT_ENV,
    "kit_dir": KIT_DIR,
    "kit_python": sys.executable,
    "kit_scratch_dir": str(WORK_DIR / "kit_scratch"),
    "image_format": IMAGE_FORMAT,
    "final_image_width": FINAL_IMAGE_WIDTH,
    "final_image_height": FINAL_IMAGE_HEIGHT,
    "png_compress_level": PNG_COMPRESS_LEVEL,
    "max_duration_s": MAX_DURATION_S,
    "layout": "3x4_rhythm",
    "n_columns": 4,
    "column_duration_s": MAX_DURATION_S / 4,
    "rhythm_lead": "II",
    "lead_layout": [["I", "aVR", "V1", "V4"], ["II", "aVL", "V2", "V5"], ["III", "aVF", "V3", "V6"]],
    "lead_order": ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"],
}
config_bytes = json.dumps(render_config, sort_keys=True).encode()
config_hash = hashlib.sha256(config_bytes).hexdigest()
lock_file = OUTPUT_DIR / ".render_config_sha256"
if lock_file.is_file() and lock_file.read_text().strip() != config_hash:
    raise RuntimeError("Existing images use different render settings. Use a fresh OUTPUT_DIR or restore the prior settings.")
lock_file.write_text(config_hash + "\n")
(OUTPUT_DIR / "config.json").write_text(json.dumps(render_config, indent=2) + "\n")
(OUTPUT_DIR / "images").mkdir(exist_ok=True)


## 6. Self-contained process worker

The worker is written from this cell into the working directory. Spawned Jupyter processes import that module once. Each batch is staged to canonical scratch WFDB `.dat` files (fmt 16, gain 1000, int16 round-trip verified — dodging the kit's `.mat` gain bug) and rendered by the real ECG-image-kit batch driver with the reference flags/env/fixed font; grey masters are recoloured to the canonical clean red grid and resized to 1024x768. Each PNG is written through a temporary file and atomic rename. Workers write disjoint image paths.


In [ ]:
WORKER_SOURCE = r'''
"""Clean ECG worker: render with the real ECG-image-kit (PLAN §6).

Each batch is staged to canonical scratch WFDB .dat files (fmt 16, gain
1000, canonical lead order, int16 round-trip verified — this dodges the
kit's .mat gain bug, PLAN §0.5) and rendered by the kit batch driver
with the reference flags/env/fixed font. Grey masters are recoloured to
the canonical clean red grid (fixed LUT, PLAN §10) and resized LANCZOS
to 994x768 + 15 px paper-colour pads -> final 1024x768 PNG (PLAN §7.3).

Self-contained: stdlib + numpy + scipy + Pillow only. The kit itself
runs in a subprocess (kit_python, which carries the kit's own deps).
"""
import math
import os
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
from scipy.io import loadmat
from PIL import Image

LEADS = ("I", "II", "III", "aVR", "aVL", "aVF",
         "V1", "V2", "V3", "V4", "V5", "V6")
KIT_SCRIPT = "gen_ecg_images_from_data_batch.py"
KIT_TIMEOUT_S = 3600

# Clean-view recolour: fixed "red" palette (PLAN §10.3; port of
# v1_support.variants._lut with ink=(20,20,20), paper=red paper, sat=1.0).
GRID_MAJOR = (178.0, 86.0, 84.0)
GRID_MINOR = (228.0, 180.0, 176.0)
GRID_PAPER = (253.0, 249.0, 247.0)
GRID_INK = (20.0, 20.0, 20.0)
GRID_TRACE_MAX = 55
GRID_MAJOR_LEVEL = 102
GRID_MINOR_LEVEL = 191
GRID_PAPER_LEVEL = 250

FINAL_W, FINAL_H = 1024, 768
RESIZE_H = 768
MASTER_W, MASTER_H = 2200, 1700

CFG = None
INPUT = None
OUTPUT = None
_LUT = None


class InvalidRecordError(ValueError):
    pass


def init_worker(config, input_dir, output_dir):
    global CFG, INPUT, OUTPUT
    CFG = config
    INPUT = Path(input_dir)
    OUTPUT = Path(output_dir)
    for key in ("kit_dir", "kit_python", "kit_seed", "kit_fixed_font",
                "kit_render_flags", "kit_env", "kit_scratch_dir",
                "max_duration_s", "png_compress_level"):
        if key not in CFG:
            raise ValueError(f"render_config missing {key!r}")
    if not (Path(CFG["kit_dir"]) / KIT_SCRIPT).is_file():
        raise ValueError("kit batch driver missing — rerun the kit-bundle cell")


def _lut():
    major = np.asarray(GRID_MAJOR, dtype=np.float32)
    minor = np.asarray(GRID_MINOR, dtype=np.float32)
    paper = np.asarray(GRID_PAPER, dtype=np.float32)
    ink = np.asarray(GRID_INK, dtype=np.float32)

    def _sat(c):
        lum = float(np.dot(c, (0.299, 0.587, 0.114)))
        return np.clip(lum + (c - lum) * 1.0, 0, 255)

    major, minor = _sat(major), _sat(minor)
    xs = np.array([0, GRID_TRACE_MAX, GRID_MAJOR_LEVEL,
                   GRID_MINOR_LEVEL, GRID_PAPER_LEVEL, 255], dtype=np.float32)
    lut = np.empty((256, 3), dtype=np.float32)
    grid = np.arange(256, dtype=np.float32)
    for ch in range(3):
        ys = np.array([ink[ch] * 0.35, ink[ch], major[ch], minor[ch],
                       paper[ch], min(255.0, paper[ch] + 3)], dtype=np.float32)
        lut[:, ch] = np.interp(grid, xs, ys)
    return np.clip(lut, 0, 255)


def recolour_clean(master):
    """Fixed clean-red recolour of a grey kit master."""
    global _LUT
    if _LUT is None:
        _LUT = _lut()
    grey = np.asarray(master.convert("RGB").convert("L"))
    return Image.fromarray(_LUT[grey].astype(np.uint8), "RGB")


def _paper_colour(arr):
    # 90th percentile of corner patches (not median): antialiased grid-halo
    # pixels drag a median toward tint; p90 recovers the paper white.
    h, w = arr.shape[:2]
    k = max(8, min(h, w) // 40)
    patches = np.concatenate([
        arr[:k, :k].reshape(-1, 3), arr[:k, -k:].reshape(-1, 3),
        arr[-k:, :k].reshape(-1, 3), arr[-k:, -k:].reshape(-1, 3)])
    return tuple(int(v) for v in np.percentile(patches, 90, axis=0))


def to_final(img):
    """LANCZOS fit to height 768 + symmetric paper-colour pad to 1024x768."""
    w, h = img.size
    scale = RESIZE_H / h
    new_w = round(w * scale)
    small = img.resize((new_w, RESIZE_H), Image.LANCZOS)
    if abs((w / h) - (new_w / RESIZE_H)) > 0.5 / RESIZE_H + 1e-9:
        raise InvalidRecordError(
            f"aspect drift: source {w}x{h} -> {new_w}x{RESIZE_H}")
    pad_total = FINAL_W - new_w
    if pad_total < 0:
        raise InvalidRecordError(
            f"resized width {new_w} exceeds final {FINAL_W}")
    left = pad_total // 2
    fill = _paper_colour(np.asarray(small.convert("RGB"), dtype=np.float32))
    canvas = Image.new("RGB", (FINAL_W, FINAL_H), fill)
    canvas.paste(small.convert("RGB"), (left, 0))
    if canvas.size != (FINAL_W, FINAL_H):
        raise InvalidRecordError(f"final size {canvas.size}")
    return canvas


def parse_header(path):
    try:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    except OSError as exc:
        raise InvalidRecordError(f"Unreadable header: {exc}") from exc
    if not lines:
        raise InvalidRecordError("Empty header")
    first = lines[0].split()
    if len(first) < 4 or "/" in first[0] or first[0] != path.stem:
        raise InvalidRecordError("Malformed/multi-segment record line")
    try:
        nleads = int(first[1])
        fs = float(first[2].split("/")[0])
        nsamp = int(first[3])
    except ValueError as exc:
        raise InvalidRecordError("Invalid record dimensions") from exc
    if nleads != 12 or not math.isfinite(fs) or fs <= 0 or nsamp <= 0:
        raise InvalidRecordError(
            f"Unsupported dimensions: {nleads} leads, fs={fs}, n={nsamp}")
    duration = nsamp / fs
    if duration > CFG["max_duration_s"] + 1e-6:
        raise InvalidRecordError(
            f"Duration {duration:.4f}s exceeds configured maximum")
    rows = []
    comments = {}
    for line in lines[1:]:
        if line.startswith("#"):
            body = line[1:].strip()
            if ":" in body:
                k, v = body.split(":", 1)
                comments[k.strip()] = v.strip()
            continue
        if not line.strip():
            continue
        f = line.split()
        if len(f) < 9:
            raise InvalidRecordError("Malformed signal line")
        name = f[0]
        if name != Path(name).name or name in (".", "..", "~"):
            raise InvalidRecordError("Unsafe signal filename")
        fmt = f[1]
        if fmt not in ("16", "16+24", "16x1+24"):
            raise InvalidRecordError(f"Unsupported signal format {fmt}")
        gain_field = f[2]
        if not gain_field.endswith("/mV"):
            raise InvalidRecordError(f"Expected /mV units, got {gain_field}")
        try:
            gain = float(gain_field.split("(")[0].split("/")[0])
            adc_zero = int(f[4])
        except ValueError as exc:
            raise InvalidRecordError("Invalid gain or ADC zero") from exc
        if not math.isfinite(gain) or gain <= 0:
            raise InvalidRecordError("Nonpositive/nonfinite ADC gain")
        if "(" in gain_field and gain_field.split("(", 1)[1].split(")", 1)[0] != str(adc_zero):
            raise InvalidRecordError("Header baseline/ADC zero disagree")
        rows.append({"file": name, "fmt": fmt, "gain": gain,
                     "zero": adc_zero, "lead": f[-1]})
    if len(rows) != 12:
        raise InvalidRecordError("Need exactly 12 signal lines")
    upper = {lead.upper(): lead for lead in LEADS}
    names = []
    for r in rows:
        key = r["lead"].strip().upper()
        if key not in upper:
            raise InvalidRecordError(f"Non-canonical lead {r['lead']!r}")
        names.append(upper[key])
    if sorted(names) != sorted(LEADS):
        raise InvalidRecordError("Lead set is not the canonical 12")
    if len({r["file"] for r in rows}) != 1 or len({r["fmt"] for r in rows}) != 1:
        raise InvalidRecordError("Multiple signal files/formats not supported")
    return rows, names, fs, nsamp, duration, comments


def load_mv(path, rows, names, nsamp):
    fmt = rows[0]["fmt"]
    sig_path = path.parent / rows[0]["file"]
    if not sig_path.is_file():
        raise InvalidRecordError(f"Missing signal file {rows[0]['file']}")
    if fmt == "16":
        n = 12 * nsamp
        raw = np.fromfile(str(sig_path), dtype="<i2", count=n).astype(np.int64)
        if raw.size != n:
            raise InvalidRecordError(
                f".dat holds {raw.size} samples, need {n}")
        raw = raw.reshape(nsamp, 12).T
    else:
        try:
            arr = np.asarray(loadmat(str(sig_path), variable_names=["val"])["val"])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            raise InvalidRecordError(f"Unreadable MAT val: {exc}") from exc
        if arr.shape == (nsamp, 12):
            arr = arr.T
        if arr.shape != (12, nsamp) or arr.dtype.kind not in "iu":
            raise InvalidRecordError(
                f"Invalid MAT val shape/type {arr.shape}/{arr.dtype}")
        raw = arr.astype(np.int64)
    if raw.shape != (12, nsamp):
        raise InvalidRecordError(f"Unexpected signal shape {raw.shape}")
    pos = {name: i for i, name in enumerate(names)}
    order = [pos[lead] for lead in LEADS]
    gains = np.array([rows[pos[lead]]["gain"] for lead in LEADS],
                     dtype=np.float64)[:, None]
    zeros = np.array([rows[pos[lead]]["zero"] for lead in LEADS],
                     dtype=np.float64)[:, None]
    # Necessary unit conversion only: digital ADC counts -> physical millivolts.
    mv = (raw[order, :].astype(np.float64) - zeros) / gains
    if not np.isfinite(mv).all():
        raise InvalidRecordError("Nonfinite physical signal")
    return mv


def stage_record(record_id, mv, fs, comments, staged_dir):
    """Write canonical scratch .dat + .hea for the kit; verify round-trip."""
    if mv.shape[0] != 12:
        raise InvalidRecordError(f"shape {mv.shape}, need (12, N)")
    n = mv.shape[1]
    raw = np.round(mv * 1000.0)
    if raw.min() < -32768 or raw.max() > 32767:
        raise InvalidRecordError("signal exceeds int16 at gain 1000")
    staged_dir.mkdir(parents=True, exist_ok=True)
    hea = staged_dir / f"{record_id}.hea"
    dat = staged_dir / f"{record_id}.dat"
    np.ascontiguousarray(raw.astype("<i2").T).tofile(str(dat))
    rec_fs = int(fs) if float(fs).is_integer() else fs
    lines = [f"{record_id} 12 {rec_fs} {n}"]
    for lead in LEADS:
        lines.append(f"{record_id}.dat 16 1000.0(0)/mV 16 0 0 0 0 {lead}")
    for key in ("Age", "Sex"):
        if comments.get(key):
            lines.append(f"# {key}: {comments[key]}")
    hea.write_text("\n".join(lines) + "\n")
    back = np.fromfile(str(dat), dtype="<i2",
                       count=12 * n).astype(np.int64).reshape(n, 12).T
    if back.shape != raw.shape or not np.array_equal(back, raw.astype(np.int64)):
        raise InvalidRecordError("staging round-trip mismatch")
    return hea


def run_kit(staged_dir, render_dir):
    """Invoke the kit batch driver once for a staged directory."""
    kit_dir = Path(CFG["kit_dir"])
    if not (kit_dir / KIT_SCRIPT).is_file():
        raise RuntimeError(f"kit script missing: {kit_dir / KIT_SCRIPT}")
    render_dir.mkdir(parents=True, exist_ok=True)
    # NOTE: absolute paths are mandatory — the kit subprocess runs with
    # cwd=kit_dir, so relative -i/-o would resolve inside the kit tree.
    cmd = [CFG["kit_python"], KIT_SCRIPT,
           "-i", str(Path(staged_dir).resolve()),
           "-o", str(Path(render_dir).resolve()),
           "-se", str(CFG["kit_seed"]), *CFG["kit_render_flags"]]
    env = dict(os.environ)
    env.update(CFG.get("kit_env", {}))
    env["ECG_FIXED_FONT"] = CFG["kit_fixed_font"]
    # one numerical thread per renderer; never the GPU
    env.update({"CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "1",
                "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
    proc = subprocess.run(cmd, cwd=str(kit_dir), capture_output=True,
                          text=True, timeout=KIT_TIMEOUT_S, env=env)
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "").strip().splitlines()[-3:]
        raise RuntimeError(f"kit batch failed "
                           f"({staged_dir} -> {render_dir}): {'; '.join(tail)}")


def collect_pngs(render_dir, record_ids):
    """Map record_id -> rendered PNG. The kit names outputs <record>-0.png."""
    found = {}
    for png in Path(render_dir).rglob("*.png"):
        stem = png.stem  # <record_id>-0
        rid = stem[:-2] if stem.endswith("-0") else stem
        if rid in record_ids:
            found[rid] = png
    missing = sorted(set(record_ids) - set(found))
    if missing:
        raise RuntimeError(f"kit produced no PNG for {len(missing)} records, "
                           f"e.g. {missing[:5]}")
    return found


def clean_render_dir(render_dir):
    """Remove kit side products; only *.png may remain."""
    for pat in ("*.hea", "*.dat", "*.json", "*.csv"):
        for p in Path(render_dir).rglob(pat):
            try:
                p.unlink()
            except OSError:
                pass


def record_id_for(rel):
    stem = Path(rel).with_suffix("").as_posix()
    rid = stem.replace("/", "__").replace("\\", "__")
    if not rid or rid in (".", "..") or "/" in rid or len(rid) > 180:
        raise InvalidRecordError(f"Unusable record id for {rel!r}")
    return rid


def _base_row(rel):
    return {"source_header": rel,
            "image_path": str(Path("images") / Path(rel).with_suffix(".png")),
            "status": "", "error": "", "sample_rate_hz": "", "samples": "",
            "duration_s": "", "image_width": "", "image_height": "",
            "image_bytes": ""}


def process_batch(batch):
    scratch = (Path(CFG["kit_scratch_dir"])
               / f"w{os.getpid()}-{int(time.time() * 1000) % 100000000}")
    staged = scratch / "staged"
    render = scratch / "render"
    rows = {}
    todo = {}
    for rel in batch:
        base = _base_row(rel)
        rows[rel] = base
        header = INPUT / rel
        try:
            prows, names, fs, nsamp, duration, comments = parse_header(header)
            base.update(sample_rate_hz=fs, samples=nsamp,
                        duration_s=round(duration, 6))
            out = OUTPUT / base["image_path"]
            if out.is_file() and out.stat().st_size > 100:
                try:
                    with Image.open(out) as existing:
                        ok = existing.size == (FINAL_W, FINAL_H)
                        existing.verify()
                    if ok:
                        base.update(status="skipped", image_width=FINAL_W,
                                    image_height=FINAL_H,
                                    image_bytes=out.stat().st_size)
                        continue
                except (OSError, ValueError):
                    pass
                # A mismatched or damaged existing image is regenerated.
            mv = load_mv(header, prows, names, nsamp)
            rid = record_id_for(rel)
            if rid in todo:
                raise InvalidRecordError(f"Duplicate staged id {rid}")
            todo[rid] = (rel, mv, fs, comments)
        except InvalidRecordError as exc:
            base.update(status="invalid", error=str(exc)[:1000])
        except Exception as exc:
            base.update(status="failed",
                        error=f"{type(exc).__name__}: {exc}"[:1000])
    try:
        if todo:
            for rid, (rel, mv, fs, comments) in todo.items():
                if rows[rel]["status"]:
                    continue
                try:
                    stage_record(rid, mv, fs, comments, staged)
                except InvalidRecordError as exc:
                    rows[rel].update(status="invalid", error=str(exc)[:1000])
            staged_ids = [rid for rid in todo if not rows[todo[rid][0]]["status"]]
            if staged_ids:
                try:
                    run_kit(staged, render)
                    got = collect_pngs(render, set(staged_ids))
                except Exception as exc:
                    for rid in staged_ids:
                        rows[todo[rid][0]].update(
                            status="failed",
                            error=f"{type(exc).__name__}: {exc}"[:1000])
                    got = {}
                for rid, png in got.items():
                    rel = todo[rid][0]
                    base = rows[rel]
                    try:
                        with Image.open(png) as master:
                            if master.size != (MASTER_W, MASTER_H):
                                raise InvalidRecordError(
                                    f"master size {master.size}, "
                                    f"need {(MASTER_W, MASTER_H)}")
                            view = recolour_clean(master)
                        final = to_final(view)
                        out = OUTPUT / base["image_path"]
                        out.parent.mkdir(parents=True, exist_ok=True)
                        temp = out.with_name(out.name + f".{os.getpid()}.part")
                        try:
                            final.save(temp, format="PNG",
                                       compress_level=CFG["png_compress_level"])
                            temp.replace(out)
                        finally:
                            try:
                                temp.unlink(missing_ok=True)
                            except OSError:
                                pass
                            final.close()
                        base.update(status="success", image_width=FINAL_W,
                                    image_height=FINAL_H,
                                    image_bytes=out.stat().st_size)
                    except InvalidRecordError as exc:
                        base.update(status="invalid", error=str(exc)[:1000])
                    except Exception as exc:
                        base.update(status="failed",
                                    error=f"{type(exc).__name__}: {exc}"[:1000])
                clean_render_dir(render)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
    return [rows[rel] for rel in batch]


def process_one(rel):
    return process_batch([rel])[0]


def ping():
    """Zero-cost import probe: lets §7 verify a spawn-context pool child
    can import this module without rendering anything."""
    return {"ok": True, "leads": list(LEADS),
            "final": (FINAL_W, FINAL_H), "master": (MASTER_W, MASTER_H)}

'''

worker_path = WORK_DIR / "ecg_clean_worker.py"
compile(WORKER_SOURCE, str(worker_path), "exec")
worker_path.write_text(WORKER_SOURCE)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
# Spawn-context pool children (§8) start a fresh interpreter: they inherit
# os.environ but NOT this sys.path edit, and both the pool initializer and
# every task import ecg_clean_worker by module reference. PYTHONPATH carries
# WORK_DIR across, otherwise every child dies with ModuleNotFoundError.
if str(WORK_DIR) not in os.environ.get("PYTHONPATH", "").split(os.pathsep):
    os.environ["PYTHONPATH"] = str(WORK_DIR) + (
        (os.pathsep + os.environ["PYTHONPATH"]) if os.environ.get("PYTHONPATH") else "")
importlib.invalidate_caches()
import ecg_clean_worker
ecg_clean_worker = importlib.reload(ecg_clean_worker)
print("Worker module ready:", worker_path)


## 7. Preflight / smoke test (run before the overnight job)

Run this cell after the worker cell and before the full generate. It takes ~1-2 min and checks everything that would otherwise wake you up at 3am: imports, kit bundle + patch gate + fixed font, AWS credentials and source reachability, **upload write permission** (tiny probe object, deleted afterwards), disk space, worker import, and one real end-to-end record through the kit (`stage -> run_kit -> recolour -> 1024x768 PNG`). The test image lands in the real `OUTPUT_DIR` so the full run reuses it as `skipped`.

Do not start §8 until this cell prints `PREFLIGHT PASS`. If it prints `PREFLIGHT FAIL`, fix the listed item(s) and re-run this cell only.



In [ ]:
# §7 preflight: fail fast before the overnight run (~1-2 min).
import time as _pf_time

_pf_failures = []
_pf_warnings = []

def _pf_check(name, fn):
    try:
        detail = fn()
        print(f"  [PASS] {name}" + (f" — {detail}" if detail else ""), flush=True)
        return detail
    except Exception as exc:
        print(f"  [FAIL] {name}: {type(exc).__name__}: {exc}", flush=True)
        _pf_failures.append(f"{name}: {type(exc).__name__}: {exc}")
        return None

print("Preflight — fast checks first ...", flush=True)

def _pf_imports():
    import boto3 as _b, numpy as _np, scipy as _sp, PIL as _pil
    from PIL import Image as _I
    import matplotlib as _mpl, pandas as _pd, qrcode as _qr, yaml as _y, wfdb as _w
    return (f"py {sys.version.split()[0]}, boto3 {_b.__version__}, "
            f"np {_np.__version__}, scipy {_sp.__version__}, "
            f"Pillow {_I.__version__}, mpl {_mpl.__version__}, "
            f"pd {_pd.__version__}, wfdb {_w.__version__}")
_pf_check("imports (boto3/numpy/scipy/Pillow/matplotlib/pandas/qrcode/yaml/wfdb)", _pf_imports)

def _pf_paths():
    assert INPUT_DIR.is_dir(), f"INPUT_DIR missing: {INPUT_DIR}"
    assert OUTPUT_DIR.is_dir(), f"OUTPUT_DIR missing: {OUTPUT_DIR}"
    drv = Path(render_config["kit_dir"]) / "gen_ecg_images_from_data_batch.py"
    assert drv.is_file(), f"kit driver missing: {drv}"
    font = Path(render_config["kit_dir"]) / "Fonts" / render_config["kit_fixed_font"]
    assert font.is_file(), f"fixed font missing: {font}"
    n = len(records)
    assert n > 0, "no records discovered — rerun §5"
    return f"{n:,} records, kit {render_config['kit_dir']}, font {render_config['kit_fixed_font']}"
_pf_check("paths (INPUT/OUTPUT/kit driver/font/records)", _pf_paths)

def _pf_patch_gate():
    need = {"ecg_plot.py": ["PATCHED-LEADNAME", "PATCHED-RHYTHMNAME", "PATCHED-INKWIDTH",
                            "PATCHED-LAYOUT", "PATCHED-PAPER"],
            "extract_leads.py": ["PATCHED-COLUMNS"],
            "gen_ecg_images_from_data_batch.py": ["PATCHED-NUMPY2"],
            "gen_ecg_image_from_data.py": ["PATCHED-LAZYDEPS", "PATCHED-FIXEDFONT"],
            "helper_functions.py": ["PATCHED-UNUSED-IMGAUG"]}
    missing = [f"{n}:{m}" for n, ms in need.items()
               for m in ms if m not in (Path(render_config["kit_dir"]) / n).read_text()]
    assert not missing, f"unpatched kit: {missing[:4]}"
    return "all 11 markers present"
_pf_check("kit patch gate", _pf_patch_gate)

def _pf_workers():
    import os as _os
    cpu = _os.cpu_count() or 0
    assert WORKERS >= 1 and BATCH_SIZE >= 1, "WORKERS/BATCH_SIZE misconfigured"
    return f"WORKERS={WORKERS} (cpu={cpu}), BATCH_SIZE={BATCH_SIZE}"
_pf_check("workers/cpu", _pf_workers)

def _pf_disk():
    du = shutil.disk_usage(WORK_DIR)
    free_gib = du.free / 2**30
    # At this point source (8.7) + extracted tree (21.9) are already on disk;
    # still needed: images (~178) + output TAR (~178) + scratch (<2).
    need_gib = 360.0
    msg = f"{free_gib:.0f} GiB free, need ~{need_gib:.0f} GiB from here"
    assert free_gib >= need_gib, msg + " — enlarge the EBS volume before §8"
    return msg
_pf_check("disk space on WORK_DIR volume", _pf_disk)

def _pf_aws_source():
    creds = aws_session.get_credentials()
    assert creds is not None, "no AWS credentials — attach the EC2 IAM role and rerun §3"
    info = s3.head_object(Bucket=src_bucket, Key=src_key)
    assert int(info["ContentLength"]) == source_bytes, "S3 source size changed since §3"
    return f"creds via {creds.method}, source {source_bytes / 2**30:.2f} GiB reachable"
_pf_check("AWS creds + source reachable", _pf_aws_source)

def _pf_upload_probe():
    up = urlparse(S3_UPLOAD_PREFIX)
    assert up.scheme == "s3" and up.netloc, f"bad S3_UPLOAD_PREFIX: {S3_UPLOAD_PREFIX!r}"
    key = "/".join(p for p in (up.path.strip("/"), ".preflight_write_probe.txt") if p)
    body = f"preflight {datetime.now(timezone.utc).isoformat()}\n".encode()
    try:
        s3.put_object(Bucket=up.netloc, Key=key, Body=body)
        head = s3.head_object(Bucket=up.netloc, Key=key)
        assert int(head["ContentLength"]) == len(body), "probe size mismatch"
        return f"write+read OK at s3://{up.netloc}/{key} (deleted after)"
    finally:
        try:
            s3.delete_object(Bucket=up.netloc, Key=key)
        except Exception:
            pass
_pf_check("upload write permission (probe object)", _pf_upload_probe)

def _pf_worker_import():
    assert hasattr(ecg_clean_worker, "process_one"), "worker has no process_one — rerun §6"
    assert hasattr(ecg_clean_worker, "init_worker"), "worker has no init_worker — rerun §6"
    return str(worker_path)
_pf_check("worker module import", _pf_worker_import)

def _pf_pythonpath():
    pp = os.environ.get("PYTHONPATH", "")
    assert str(WORK_DIR) in pp.split(os.pathsep), \
        f"WORK_DIR not on PYTHONPATH ({pp!r}) — rerun §6"
    return "WORK_DIR visible to spawn children"
_pf_check("PYTHONPATH carries WORK_DIR to pool children", _pf_pythonpath)

print("Preflight — spawn-pool child import (same machinery as §8) ...", flush=True)
try:
    _pf_ctx = mp.get_context("spawn")
    with ProcessPoolExecutor(max_workers=1, mp_context=_pf_ctx) as _pf_ex:
        _pf_pong = _pf_ex.submit(ecg_clean_worker.ping).result(timeout=300)
    assert _pf_pong.get("ok") and tuple(_pf_pong.get("final", ())) == (FINAL_IMAGE_WIDTH, FINAL_IMAGE_HEIGHT), \
        f"bad pong: {_pf_pong}"
    print(f"  [PASS] spawn-pool ping — child imported worker, pong {_pf_pong}", flush=True)
except Exception as exc:
    print(f"  [FAIL] spawn-pool ping: {type(exc).__name__}: {exc}", flush=True)
    _pf_failures.append(f"spawn-pool ping: {type(exc).__name__}: {exc}")

print("Preflight — one real record end-to-end ...", flush=True)
_pf_single_secs = None
_pf_single_rel = None
try:
    ecg_clean_worker.init_worker(render_config, str(INPUT_DIR), str(OUTPUT_DIR))
    _pf_row = None
    for _rel in list(records[:3]):
        t0 = _pf_time.monotonic()
        _cand = ecg_clean_worker.process_one(_rel)
        dt = _pf_time.monotonic() - t0
        if _cand.get("status") in ("success", "skipped"):
            _pf_row, _pf_single_secs, _pf_single_rel = _cand, dt, _rel
            break
        print(f"  [WARN] {_rel} -> {_cand.get('status')}: {_cand.get('error','')[:160]}", flush=True)
    assert _pf_row is not None, "first 3 records all failed — inspect §5 data before §8"
    out = OUTPUT_DIR / _pf_row["image_path"]
    assert out.is_file() and out.stat().st_size > 50_000, f"test PNG missing/small: {out}"
    with Image.open(out) as _im:
        assert _im.size == (FINAL_IMAGE_WIDTH, FINAL_IMAGE_HEIGHT), f"test PNG size {_im.size}"
    est_h = _pf_single_secs * len(records) / max(WORKERS, 1) / 3600
    print(f"  [PASS] single-record render: {_pf_single_rel} -> {out} "
          f"({out.stat().st_size / 2**20:.2f} MiB, {_pf_single_secs:.0f}s; "
          f"est. full run ~{est_h:.1f}h on {WORKERS} workers)", flush=True)
    try:
        from IPython.display import display as _display
        _display(Image.open(out))
    except Exception:
        pass
except Exception as exc:
    print(f"  [FAIL] single-record render: {type(exc).__name__}: {exc}", flush=True)
    _pf_failures.append(f"single-record render: {type(exc).__name__}: {exc}")

print(f"Preflight done: {len(records):,} records, "
      f"{split_counts.get('train', 0):,}/{split_counts.get('validation', 0):,}/{split_counts.get('test', 0):,} "
      f"(train/val/test).", flush=True)
if _pf_failures:
    raise RuntimeError("PREFLIGHT FAIL — fix and re-run this cell only:\n- " + "\n- ".join(_pf_failures))
print("PREFLIGHT PASS — safe to run §8 (generate) overnight.", flush=True)


## 8. Generate images and reports

At most `MAX_PENDING_BATCHES` are queued, so neither the task queue nor results grow with the full dataset. Status CSVs are written by the parent only; workers write disjoint image paths. If some records are corrupt, the job continues and their paths/reasons are recorded. `processed = successful + skipped + invalid + failed` in the final summary. `successful` counts newly written images, while `skipped` counts valid images reused from an earlier run.

Only run after the §7 preflight cell prints `PREFLIGHT PASS`.



In [ ]:
from collections import Counter

# Spawn-context children inherit os.environ, not sys.path: re-ensure WORK_DIR
# is on PYTHONPATH (normally set in §6) so children can import ecg_clean_worker.
if str(WORK_DIR) not in os.environ.get("PYTHONPATH", "").split(os.pathsep):
    os.environ["PYTHONPATH"] = str(WORK_DIR) + (
        (os.pathsep + os.environ["PYTHONPATH"]) if os.environ.get("PYTHONPATH") else "")

fields = ["source_header", "image_path", "status", "error", "sample_rate_hz",
          "samples", "duration_s", "image_width", "image_height", "image_bytes"]
manifest_path = OUTPUT_DIR / "manifest.csv"
failures_path = OUTPUT_DIR / "failures.csv"
counts = Counter()
image_bytes_total = 0
started = time.monotonic()
last_report = 0

def batches(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]

def record_result(row, manifest_writer, failure_writer):
    global image_bytes_total
    counts[row["status"]] += 1
    image_bytes_total += int(row["image_bytes"] or 0)
    manifest_writer.writerow(row)
    if row["status"] in {"invalid", "failed"}:
        failure_writer.writerow(row)

with manifest_path.open("w", newline="") as manifest_file, failures_path.open("w", newline="") as failure_file:
    manifest_writer = csv.DictWriter(manifest_file, fieldnames=fields)
    failure_writer = csv.DictWriter(failure_file, fieldnames=fields)
    manifest_writer.writeheader()
    failure_writer.writeheader()

    batch_iter = iter(batches(records, BATCH_SIZE))
    context = mp.get_context("spawn")
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=context,
                             initializer=ecg_clean_worker.init_worker,
                             initargs=(render_config, str(INPUT_DIR), str(OUTPUT_DIR))) as pool:
        pending = {}
        for _ in range(MAX_PENDING_BATCHES):
            batch = next(batch_iter, None)
            if batch is None:
                break
            pending[pool.submit(ecg_clean_worker.process_batch, batch)] = batch
        while pending:
            done, _ = wait(pending, return_when=FIRST_COMPLETED)
            for future in done:
                batch = pending.pop(future)
                try:
                    result_rows = future.result()
                except Exception as exc:
                    result_rows = [{"source_header": rel,
                                    "image_path": (Path("images") / Path(rel).with_suffix(".png")).as_posix(),
                                    "status": "failed", "error": f"Worker batch failure: {type(exc).__name__}: {exc}"[:1000],
                                    "sample_rate_hz": "", "samples": "", "duration_s": "",
                                    "image_width": "", "image_height": "", "image_bytes": ""}
                                   for rel in batch]
                for row in result_rows:
                    record_result(row, manifest_writer, failure_writer)
                next_batch = next(batch_iter, None)
                if next_batch is not None:
                    pending[pool.submit(ecg_clean_worker.process_batch, next_batch)] = next_batch
            processed = sum(counts.values())
            if processed - last_report >= PROGRESS_EVERY or processed == len(records):
                elapsed = max(time.monotonic() - started, 1)
                print(f"{processed:,}/{len(records):,} ({processed / elapsed:.1f}/s) | "
                      f"new {counts['success']:,}, reused {counts['skipped']:,}, "
                      f"invalid {counts['invalid']:,}, failed {counts['failed']:,}", flush=True)
                manifest_file.flush()
                failure_file.flush()
                last_report = processed

summary = {
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "source": S3_SOURCE,
    "discovered": len(records),
    "processed": sum(counts.values()),
    "successful": counts["success"],
    "skipped": counts["skipped"],
    "invalid": counts["invalid"],
    "failed": counts["invalid"] + counts["failed"],
    "runtime_failures": counts["failed"],
    "image_bytes": image_bytes_total,
    "elapsed_seconds": round(time.monotonic() - started, 1),
    "render_config_sha256": config_hash,
}
if summary["processed"] != len(records):
    raise RuntimeError("Generation did not account for every discovered header")
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))
print("Manifest:", manifest_path, "| failures:", failures_path)


## 9. Package and upload

The output archive contains `images/train/`, `images/validation/`, and `images/test/`, plus the source patient and record manifests, rendering manifest, failure log, configuration, and summary. PNG is already losslessly compressed, so plain TAR is the fast default; set `ARCHIVE_FORMAT = "tar.gz"` before running for gzip compression. The archive is built to a temporary path and atomically renamed. The upload is checked against S3 `ContentLength`. Review `summary.json` and `failures.csv` before using any images for training.


In [ ]:
archive_path = WORK_DIR / f"{ARCHIVE_BASENAME}.{ARCHIVE_FORMAT}"
temp_archive = archive_path.with_name(archive_path.name + ".part")
temp_archive.unlink(missing_ok=True)
mode = "w" if ARCHIVE_FORMAT == "tar" else "w:gz"
print("Building", archive_path, "...")
with tarfile.open(temp_archive, mode, **({"compresslevel": 1} if mode == "w:gz" else {})) as archive:
    archive.add(OUTPUT_DIR, arcname=OUTPUT_DIR.name, recursive=True)
temp_archive.replace(archive_path)

# Verify listing and expected image count before upload; this reads archive metadata,
# not image pixels. TAR.GZ listing requires a full decompression pass.
with tarfile.open(archive_path, "r:*") as archive:
    packaged_images = sum(member.isfile() and member.name.startswith(OUTPUT_DIR.name + "/images/")
                          and member.name.endswith(".png") for member in archive)
expected_images = summary["successful"] + summary["skipped"]
if packaged_images != expected_images:
    raise RuntimeError(f"Archive has {packaged_images:,} images; expected {expected_images:,}")

upload_prefix = urlparse(S3_UPLOAD_PREFIX)
if upload_prefix.scheme != "s3" or not upload_prefix.netloc:
    raise ValueError("S3_UPLOAD_PREFIX must be s3://bucket/ or s3://bucket/prefix/")
upload_bucket = upload_prefix.netloc
prefix_key = upload_prefix.path.strip("/")
upload_key = "/".join(part for part in (prefix_key, archive_path.name) if part)
s3.upload_file(str(archive_path), upload_bucket, upload_key,
               Config=TransferConfig(max_concurrency=8, use_threads=True))
remote = s3.head_object(Bucket=upload_bucket, Key=upload_key)
if int(remote["ContentLength"]) != archive_path.stat().st_size:
    raise IOError("Uploaded archive size differs from local archive")
uploaded_uri = f"s3://{upload_bucket}/{upload_key}"
print(f"Uploaded {packaged_images:,} images; {archive_path.stat().st_size / 2**30:.2f} GiB -> {uploaded_uri}")
print("Final status:", "complete" if summary["failed"] == 0 else "complete with record failures; inspect failures.csv")


## Run verification

After the last cell, confirm `summary.json` has `processed == discovered`, inspect `failed` and `failures.csv`, and confirm the packaged image count equals `successful + skipped`. The source `record_manifest.csv` and `patient_assignments.csv` are included in the image archive so every PNG can be traced to its ECG and patient. Check the S3 URI and byte count printed after upload. Rerunning the notebook reuses the SHA-256 verified source archive, extracted tree, and valid PNGs. If you change rendering settings, choose a new output directory and archive basename so different renderings cannot mix.

## Storage planning

The patient-split source archive is **8.72 GiB** and contains **193,924 records** (all 500 Hz after the upstream upsampling step). The extracted WFDB tree is about **21.9 GiB**. A 3-record sample test through this notebook's worker (real `.dat`@500 Hz, `.mat`@500 Hz, `.dat`@250 Hz records) produced final 1024x768 clean PNGs of **0.92-0.96 MiB** (mean ~0.94 MiB), so the full image set projects to about **178 GiB**. PNGs are already compressed, so the output TAR is the same size again. Plan concurrent room for source archive + extracted tree + images + output archive: 8.7 + 21.9 + 178 + 178 ~= **388 GiB**, plus metadata/scratch (< 2 GiB). A **500 GB gp3 EBS volume** is the recommended starting point (~25% headroom). At the measured steady-state pace (~2.6 s/record across 14 workers) the full render takes roughly **10-14 h**, plus packaging and the ~178 GiB S3 upload. Check available space before the packaging cell.
